# rhyme-plan-hybrid: newline rhyme planning + state-vs-KV locus in Qwen3.5-0.8B (Gated-DeltaNet hybrid)
**Branch:** `feature/rhyme-plan-hybrid` | **Gate:** pre-registered in README.md (frozen 2026-11-02)
**One-Click Execution:** Runtime -> Run all. Clones, installs GDN kernels, runs pytest on GPU (aborts if patching is
broken on the kernel path), runs the resumable campaign with results on Google Drive, prints RESULTS.md, unassigns the GPU.
Re-running after a disconnect resumes from Drive (finished batches are skipped).

In [ ]:
# Step 1: Credit Guard (auto-unassign on any error or completion) + Colab Secrets
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print("\n[Credit Guard] Error, interrupt, or completion! Flushing Drive and unassigning GPU...")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

def custom_exc_handler(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    auto_teardown()
    return []

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), custom_exc_handler)

try:
    from google.colab import userdata
    for k in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "hf_token"]:
        try:
            tok = userdata.get(k)
            if tok:
                os.environ["HF_TOKEN"] = tok
                os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
                print(f"[Secrets] Loaded Hugging Face token from Colab Secrets ({k}).")
                break
        except Exception:
            pass
except ImportError:
    pass

print("[Ready] Credit guard active. VM will self-terminate on error or completion.")

In [ ]:
# Step 2: GPU check + clone repository
import torch
assert torch.cuda.is_available(), "Switch Colab runtime to GPU (L4 24GB): Runtime -> Change runtime type -> GPU (L4)"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB | torch {torch.__version__} | CUDA {torch.version.cuda}")

repo_url = "https://github.com/jaimejarauta/decoupled-delta-net.git"
branch = "feature/rhyme-plan-hybrid"
target_dir = "/content/rhyme-plan-hybrid"
if not os.path.exists(target_dir):
    subprocess.run(["git", "clone", "-b", branch, repo_url, target_dir], check=True)
else:
    subprocess.run(["git", "-C", target_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "reset", "--hard", f"origin/{branch}"], check=True)
os.chdir(target_dir)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Step 3: Installs. pronouncing+cmudict are vendored (vendor/). GDN kernels: flash-linear-attention (pure Triton, fast
# install) + causal-conv1d (CUDA extension: prebuilt wheel if one matches this torch/CUDA, else a source build; capped
# at 25 min, falls back to the torch conv path, which is correct but slower).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-linear-attention==0.6.0"], check=True)
try:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "causal-conv1d==1.7.0"], timeout=1500)
    if r.returncode:
        print("[Kernels] causal-conv1d install failed -> torch conv fallback (correct, slower).")
except subprocess.TimeoutExpired:
    print("[Kernels] causal-conv1d build timed out -> torch conv fallback (correct, slower).")
if os.environ.get("HF_TOKEN"):
    import huggingface_hub
    huggingface_hub.login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
# fresh interpreter, so it sees the new packages
print(subprocess.run([sys.executable, "-c", "import rp, transformers; print('transformers', transformers.__version__, '| kernels', rp.kernel_info(), '| device', rp.DEV)"],
                     capture_output=True, text=True, check=True).stdout)

In [ ]:
# Step 4: Mount Drive; results (incl. resume state) live there (falls back to the VM disk if the mount fails)
def mount_drive():
    from google.colab import drive
    for force in (False, True):
        try:
            drive.mount("/content/drive", force_remount=force)
            return True
        except Exception as e:
            print(f"[Drive] mount failed (force_remount={force}): {e}")
    print("[Drive] NOT mounted -> results stay on the VM only (lost if the runtime disconnects). "
          "Fix: accept the 'Permit this notebook to access your Google Drive' popup, then re-run this cell.")
    return False
os.environ["RP_RESULTS"] = ("/content/drive/MyDrive/moonshot_results/rhyme-plan-hybrid" if mount_drive()
                            else "/content/rhyme_results")
os.makedirs(os.environ["RP_RESULTS"], exist_ok=True)
print("Results dir:", os.environ["RP_RESULTS"], "| existing:", sorted(os.listdir(os.environ["RP_RESULTS"])))


In [ ]:
# Step 5: Pre-flight on the GPU kernel path (fp32): no-op patch leaves logits unchanged; BOTH patch reproduces the
# source; patch sites = 18 GDN + 6 attention. If it fails WITH the fast kernels, they are uninstalled and the checks
# re-run on the torch reference path. ABORTS (and unassigns) if that fails too. Full pytest output is PRINTED and saved
# to Drive (preflight_*.log) so a failure can be diagnosed.
def preflight(tag):
    r = subprocess.run([sys.executable, "-m", "pytest", "-v", "-x", "-p", "no:cacheprovider", "--tb=long", "test_rp.py"],
                       capture_output=True, text=True)
    out = r.stdout + r.stderr
    print(f"===== pytest [{tag}] exit={r.returncode} =====\n" + out[-8000:])
    open(os.path.join(os.environ["RP_RESULTS"], f"preflight_{tag}.log"), "w").write(out)
    return r.returncode
if preflight("fast"):
    print("[Pre-flight] FAILED on the fast-kernel path -> uninstalling FLA/causal-conv1d, retrying on the torch path")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "flash-linear-attention", "fla-core", "causal-conv1d"])
    if preflight("torch"):
        raise RuntimeError("pytest failed on GPU: cache patching invalid -> campaign aborted (see log above / preflight_*.log in Drive)")
print("[Pre-flight] all patching checks passed on GPU. Kernels:",
      subprocess.run([sys.executable, "-c", "import rp; print(rp.kernel_info())"], capture_output=True, text=True).stdout)


In [ ]:
# Step 6: Full campaign (resumable): Qwen3.5-0.8B run -> logit -> Qwen2.5-0.5B-Instruct run -> logit -> gate analysis
p = subprocess.Popen([sys.executable, "-u", "run_all.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "Loading weights" not in line:
        print(line, end="")
if p.wait():
    raise RuntimeError(f"run_all.py failed (exit {p.returncode}); re-run the notebook to resume from Drive")

In [ ]:
# Step 7: Results
print(open(os.path.join(os.environ["RP_RESULTS"], "RESULTS.md")).read())

In [ ]:
# Step 8: Flush Drive & unassign GPU
auto_teardown()